In [1]:
import emodeconnection as emc
import numpy as np
import matplotlib.pyplot as plt
import time


# Fixed parameters of Geometry
wavelength_fund = 450.0
w_core = 400.0  # [nm] Fixed width
h_core = 350.0  # [nm] Fixed height
dx, dy = 10.0, 10.0


# Sellmeier Material Definition
# Use double asterisks for power, ensure no spaces in the inner math, and verify the negative B3/C3 values are handled cleanly.
eq_o = '(1+2.8032/(1-0.015287/x**2)+0.36335/(1-0.036095/x**2)-33508000/(1+367200000/x**2))**0.5'
eq_e = '(1+0.017061/(1-0.043855/x**2)+3.1976/(1-0.022642/x**2)-57269000/(1-74226000/x**2))**0.5'
anisotropic_equation = f"[{eq_o},{eq_e},{eq_o}]"

In [2]:
# Launch EMode 
em = emc.EMode(simulation_name='example_AlN_waveguide_sweep1')



In [3]:
# Set Up Simulation
em.add_material(name='custom_AlN',                                  
                refractive_index_equation=anisotropic_equation, 
                wavelength_unit='um')
em.settings(window_width=2000,window_height=h_core+2000, boundary_condition='TM') # Using your 'TM' preference)
em.shape(name='Substrate', material='Al2O3', height=1000)
em.shape(name='core', material='custom_AlN', height=h_core, mask=w_core, etch_depth=h_core, sidewall_angle=5)
em.shape(name='TopClad', material='SiO2', height=800, shape_type='conformal')
em.plot()


'ok'

In [4]:
# Calculate Modes for FUNDAMENTAL WAVELENGTH
Nmodes_to_calculate = 10
em.settings(wavelength=wavelength_fund, x_resolution=dx, y_resolution=dy,window_width=2000,
            window_height=h_core+2000, num_modes=Nmodes_to_calculate, boundary_condition='TM') # Using your 'TM' preference)
em.FDM()                            #run the finite difference mode solver to find the modes of the structure
em.confinement(shape_list='core')   #calculate the confinement factor for each mode
em.report()                         #print information about the calculation results to command line
em.label_profile(name = 'dataset1') #store this set of results under label '0'
em.plot()


'ok'

In [6]:
## Run wavelength sweep:  FUNDAMENTAL WAVELENGTH
wav_nm = np.arange(400, 500, 10)
data = em.sweep(key = 'wavelength', values = wav_nm,
    result = ['effective_index','mode_order'])
em.save(simulation_name='example_AlN_waveguide_sweep1.eph')  # Save the simulation state to a fileem.save(simulation_name='example_AlN_waveguide_copy.eph')  # Save the simulation state to a file

'ok'

In [7]:
print(data['mode_order'])

None


In [ ]:
neff_matrix = np.array(data['effective_index'])

#Create the plot
plt.figure(figsize=(10, 6), dpi=100)

# Option A: Plot all modes found in the sweep
num_modes_found = neff_matrix.shape[1]
for i in range(num_modes_found):
    plt.plot(wav_nm, neff_matrix[:, i], '-o', label=f'Mode {i+1}')

# 3. Formatting the chart
plt.title(f'Wavelength Sweep: Effective Index vs. Wavelength\n({w_core}nm x {h_core}nm AlN Core)', fontsize=12)
plt.xlabel('Wavelength (nm)', fontsize=11)
plt.ylabel('Effective Index (n_eff)', fontsize=11)
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.legend(loc='best')

# 4. Display or save
plt.tight_layout()
plt.show()

In [11]:
print(wavelength_fund/2)

225.0


In [5]:
# Calculate Modes for SECOND HARMONIC WAVELENGTH
Nmodes_to_calculate = 40
em.settings(num_modes=Nmodes_to_calculate, wavelength=wavelength_fund/2, x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)
em.FDM()                            #run the finite difference mode solver to find the modes of the structure
em.confinement(shape_list='core')   #calculate the confinement factor for each mode
em.report()                         #print information about the calculation results to command line
em.label_profile(name = 'dataset2') #store this set of results under label 'dataset2'
em.plot()

'ok'

In [20]:
em.plot()

'ok'

In [6]:
## Run wavelength sweep:  SECOND HARMONIC WAVELENGTH
Nmodes_to_calculate = 40

## --- Step 1 & 2: Custom Tracking Sweep with Overlap ---
wav2_nm = np.arange(460, 440, -2) / 2
num_wavs = len(wav2_nm)

# Storage for results
tracked_neff = np.zeros(num_wavs)
tracked_overlaps_with_fund = np.zeros(num_wavs)
tracked_mode_indices = np.zeros(num_wavs, dtype=int)


In [14]:
#  ( OPTIONAL 
# 1. Initialization: Check the "Target" mode from the first WL
# Assuming you've already run FDM at wav2_nm[0] and identified the mode
em.settings(wavelength=wav2_nm[0], num_modes=Nmodes_to_calculate, max_effective_index=2.6)
em.FDM()
em.plot() # Visual check
# Label the TARGET mode to use for the first overlap check



'ok'

In [17]:
## SWEEP over WAVELENGTH 

# Note:  We also need the fundamental TM00 profile for the SHG overlap calculation
# (Assumes 'dataset1' was saved earlier with the 450nm results)
# Note: EMode usually handles the wavelength difference in overlap automatically

Nmodes_to_calculate = 32
fund_overlap_list = []


em.label_profile(name='reference_sh_mode')      #label the most recently calculated mode list
reference_mode_idx = 30                 

print(f"\n{'Wavelength':<12} | {'Mode Index':<10} | {'n_eff':<10} | {'Fund Overlap'}")
print("-" * 55)
for i, wl in enumerate(wav2_nm):
    # Update settings and solve for *NEW MODES*
    em.settings(wavelength=wl, num_modes=Nmodes_to_calculate, x_resolution=dx/2, y_resolution=dy/2)
    em.FDM()
    em.label_profile(name='new_sh_modes')

    # A. Find which of the 40 new modes matches our 'reference_sh_mode' best
    # em.overlap returns a list of overlap values between a labeled profile and current modes
    overlap_results = []  #define a list
    for mode_num_SHG in range(Nmodes_to_calculate):
        overlap = em.overlap(label_a='new_sh_modes', mode_a=mode_num_SHG, label_b='reference_sh_mode',mode_b=reference_mode_idx)    
        #print('Power overlap SHG mode %d + RefMode: %0.10f %%' % (mode_num_SHG, overlap*100))
        #save the overlap results to a file or data structure as needed
        overlap_results.append(overlap)
    best_match_idx = np.argmax(np.abs(overlap_results)) 
      
    # B. Calculate spatial overlap with the Fundamental TM00 (saved as 'dataset1')
    # This is useful for SHG efficiency tracking
    fund_overlap_val = em.overlap(label_a='new_sh_modes', mode_a=best_match_idx, label_b='dataset1',mode_b=0)    

    # C. Store Data
    tracked_mode_indices[i] = best_match_idx
    tracked_overlaps_with_fund[i] = fund_overlap_val
    current_neff = em.get("effective_index")[best_match_idx]
    tracked_neff[i] = current_neff
    
    # D. UPDATE REFERENCE: The current best match becomes the reference for the NEXT step
    # This "walking" reference is what allows tracking through geometry shifts
    em.label_profile(name='reference_sh_mode')
    reference_mode_idx = best_match_idx

    print(f"{wl:<12.2f} | {best_match_idx:<10} | {current_neff:<10.5f} | {fund_overlap_val:<10.5f}")

## --- Export Results ---
export_data = np.column_stack((wav2_nm, tracked_neff, tracked_overlaps_with_fund, tracked_mode_indices))
header = 'Wavelength_nm,n_eff,Overlap_with_Fund,Original_Mode_Index'
np.savetxt('Tracked_SHG_Modes.csv', export_data, delimiter=',', header=header, comments='')



Wavelength   | Mode Index | n_eff      | Fund Overlap
-------------------------------------------------------
230.00       | 30         | 2.02744    | 0.00592   
229.00       | 30         | 2.04061    | 0.00579   
228.00       | 30         | 2.05412    | 0.00566   
227.00       | 30         | 2.06798    | 0.00553   
226.00       | 30         | 2.08221    | 0.00541   
225.00       | 30         | 2.09685    | 0.00530   
224.00       | 30         | 2.11191    | 0.00518   
223.00       | 30         | 2.12745    | 0.00507   
222.00       | 30         | 2.14351    | 0.00497   
221.00       | 30         | 2.16017    | 0.00486   


In [6]:
wls = np.arange(230,220,-1)
find_phase_match_wl(400, 350, wls, 30, 0, n_modes=35)

np.float64(221.0)

In [ ]:
## FUNCTION FOR  PHASE MATCH WAVELENGTH
def find_phase_match_wl(w, h, wl_range, target_mode_shg_init, flag_do_tracking, n_modes=40):
    """
    Sweeps wavelength to find the zero-crossing of Delta_n.
    Returns the wavelength where phase matching occurs.
    """
    diffs = []
    # Track the mode across the wl_range to keep it consistent
    current_ref_idx = target_mode_shg_init

    for wl in wl_range:
        # [1] Solve Fundamental (approx. Mode 0)
        # Note: Reduced num_modes to 1 for fundamental to save time
        em.settings(wavelength=wl*2, num_modes=1, window_height=h+2000)
        
        # Redraw core for current geometry
        em.shape(name='core', material='custom_AlN', height=h, mask=w, 
                 etch_depth=h, sidewall_angle=5)
        em.FDM()
        n_fund = em.get("effective_index")[0]

        # [2] Solve SHG (Tracked Mode)
        em.settings(wavelength=wl, num_modes=n_modes)
        em.FDM()

        if flag_do_tracking:
            # Requires a previously labeled 'reference_sh_mode' to exist before the loop
            em.label_profile(name='new_sh_modes')
            overlap_results = []
            for m in range(n_modes):
                ov = em.overlap(label_a='new_sh_modes', mode_a=m, 
                                label_b='reference_sh_mode', mode_b=current_ref_idx)
                overlap_results.append(ov)
            best_match_idx = np.argmax(np.abs(overlap_results))
            
            # Update the reference for the NEXT wavelength step
            em.label_profile(name='reference_sh_mode') 
            current_ref_idx = best_match_idx
        else:
            # Faster: assume the mode index doesn't change
            best_match_idx = current_ref_idx

        n_shg = em.get("effective_index")[best_match_idx]
        diffs.append(n_fund - n_shg)
        
    # Use numpy to find the zero crossing (linear interpolation)
    # Note: interp expects x-values to be increasing. If diffs are decreasing, 
    # you may need: np.interp(0, np.sort(diffs), wl_range[np.argsort(diffs)])
    pm_wl = np.interp(0, np.sort(diffs), wl_range[np.argsort(diffs)])
    return pm_wl





In [ ]:
# Example: Wavelength in col 1, Neff in col 2
data_to_export = np.column_stack((wav2_nm, neff2_matrix[:, :]))

# Save with a header
np.savetxt(
    'AlN_Sweep_Results.csv',           # Filename
    data_to_export,                    # The array
    delimiter=',',                    # Use commas for CSV
    header='Wavelength_nm,n_eff_TE00', # Column titles
    comments=''
)

In [7]:
em.close()